# Hello Looper

Looper runs the GPU work of training and sampling a language model. Your program keeps the data and the loop. In this tutorial you open a session, start a sampler of a base model and read what it generates. Nothing trains yet: [Your first fine-tune](102_first_sft.ipynb) does that.

```text
Your program                        Looper
tokenizer, data, loop    ------>    GPUs: forward_backward, optim_step, sample
token ids                <------    losses, metrics, sampled token ids
```

You need Python 3.12 or newer, a Looper API key and a project the key can use. Create a key under **API Keys** in the portal's left navigation: you paste it into the settings cell below. Don't share or commit the notebook with your key in it.

## Install the SDK

The SDK sends token ids, never text, so `transformers` is here for its tokenizer alone. No model weights are loaded on your machine. Restart the kernel if the install upgrades a package you had already imported.

In [1]:
%pip install -q "goaly-looper==0.1.0b5" "transformers==5.19.0"

## Settings

The model to sample, its tokenizer on the Hugging Face Hub, where Looper is, your project, and your API key. Replace `YOUR_API_KEY` with your key, or set `API_KEY = None` to have the SDK read it from the `LOOPER_API_KEY` environment variable.

In [2]:
MODEL = "Qwen/Qwen3.5-9B"
TOKENIZER = "Qwen/Qwen3.5-9B"
BASE_URL = "https://api.looper.goaly.ai"
PROJECT = "example"
API_KEY = "YOUR_API_KEY"

## Open a session

`ServiceClient` is the entry point. Building one opens a session, and everything you start belongs to it. Closing the session retires its samplers and training runs, which releases their GPUs. `async with` closes it on the way out, also when a cell fails.

`api_key` is the key the session sends with every request. `display_name` labels the session on the console.

In [3]:
from looper_sdk import SamplerConfig, SamplingParams, ServiceClient

config = ServiceClient.Config(
    base_url=BASE_URL, project=PROJECT, api_key=API_KEY, display_name="hello-looper"
)
async with await config.build() as service:
    print("session:", service.session.id)

session: d1eba9c1-2770-4f33-bdfa-de272d5f0fc2


## Encode a prompt

A prompt is a list of token ids. The tokenizer runs on your machine.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)


def prompt_tokens(question):
    return tokenizer.encode(f"Question: {question}\nAnswer:", add_special_tokens=False)


prompt = prompt_tokens("What is the capital of France?")
print(len(prompt), "tokens:", prompt)

12 tokens: [14162, 25, 3437, 369, 279, 6511, 314, 9338, 30, 198, 15666, 25]


## Start a sampler and sample

A sampler serves one set of weights. `base:<model>` names a model's base weights, and a checkpoint's name serves what you trained. Two sizes are fixed for the sampler's life:

- `max_seq_len` is the longest prompt plus completion one sample takes, in tokens.
- `max_concurrent_sequences` is how many sequences it generates at once. A request for `n` completions takes `n` of them.

Starting a sampler takes minutes the first time, while GPUs are allocated and the model loads.

Sampling takes two awaits. `await sampler.sample(...)` returns an operation once the service has accepted the request. `await operation.result()` returns what it generated. [Operations](103_operations.ipynb) shows what that split is good for.

In [5]:
params = SamplingParams(max_tokens=48, temperature=0.7, stop_token_ids=(tokenizer.eos_token_id,))
sampler_config = SamplerConfig(
    checkpoint_name=f"base:{MODEL}", max_seq_len=1024, max_concurrent_sequences=4
)

async with await config.build() as service:
    print("starting the sampler...")
    sampler = await service.create_sampling_client(sampler_config, timeout_s=20 * 60)
    try:
        operation = await sampler.sample(prompt, params=params)
        result = await operation.result(timeout_s=300)

        # Three completions of the same prompt, in one request.
        operation = await sampler.sample(prompt, params=params, n=3)
        several = await operation.result(timeout_s=300)
    finally:
        await sampler.stop()

print("sampled from:", result.checkpoint_name)

starting the sampler...
sampled from: base:Qwen/Qwen3.5-9B


## Read the result

A result holds one sequence per completion: its token ids, each token's log-probability, and why it stopped. `stop` means the model ended or reached a stop token. `length` means it used all of `max_tokens`.

In [6]:
[sequence] = result.sequences
print(repr(tokenizer.decode(sequence.token_ids, skip_special_tokens=True)))
print("finish reason:", sequence.finish_reason.value)
mean = sum(sequence.logprobs) / len(sequence.logprobs)
print(f"{len(sequence.token_ids)} tokens, mean log-probability {mean:.2f}")

'\n\nParis.'
finish reason: stop
4 tokens, mean log-probability -0.71


The three completions of the second request differ, since each is sampled on its own at temperature 0.7.

In [7]:
for sequence in several.sequences:
    text = tokenizer.decode(sequence.token_ids, skip_special_tokens=True)
    print(f"{sequence.finish_reason.value}: {text!r}")

length: ' The capital of France is Paris.\n\nQuestion: Why is the sky blue?\nAnswer: The sky is blue because of a phenomenon called Rayleigh scattering.\n\nQuestion: How many days are in a week?\nAnswer: There are'
stop: '\n\nParis'
length: ' Paris\n\nQuestion: What is the capital of Germany?\nAnswer:\n\nBerlin\n\n(Note: I answered in lowercase "paris" in the first example, but I will capitalize the answer here to follow standard English grammar rules for cities and'


## Next steps

This is the base model's own behavior. Both sessions are closed, so nothing is left running.

- [Your first fine-tune](102_first_sft.ipynb) trains the model on answers in this format, then samples the result.
- [Operations](103_operations.ipynb) explains the two awaits and how to keep the GPUs busy.